# Exploring the YouTube Data API

**Project:** YouTube Title Predictor (ML Zoomcamp midterm)
**Purpose:** Testing the API in a notebook lets me check if it is returning the information I need and that its in the right fromat.

This is a scratch notebook for learning how the API responds before I build
the real data collection script. The graded analysis lives in `notebook.ipynb`.

## What's in here
1. Load the API key from `.env` (never printed)
2. The call asks for 3 parts: `snippet`, `contentDetails` and `statistics`
3. Pulled out from the json response was the `Channel ID`, `Channel title`, `Subscriber count`, `Video count` and ` Uploads playlist`. This will later be used to create a dataset. 

## Notes / findings
- `subscriberCount` comes back as strings, so they need to be converted to numbers before doing any analysis.
- The upload playlist ID is the channel ID with the second letter changed from `C` to `U` (`UCqxY...`  → `UUqxY...`); everything after that is identical.  Google doesn't officially document this, so `collect_data.py` will read the ID from `contentDetails.relatedPlaylists.uploads` rather than build it. The pattern is still a handy sanity check.
- This call costs 1 quota unit (out of 10,000 a day).
- only 3 of the seed channels fall within the 1k-100k subscribers, well-known channels tend to be too big. The list will be topped up with `search.list`.
- `search.list` only supports `part="snippet"`, so it returns no subscriber
  counts. A follow-up `channels.list` call is needed to get those.
- In search results, `id` is a dict, not a string: the channel ID is at
  `item["id"]["channelId"]` (in `channels.list` it's just `item["id"]`).
- Each `search.list` call costs 100 quota units and returns at most 50 results.
  The response includes a `nextPageToken` when more results exist, which can
  be sent back to get the next page (another 100 units).
- Test query "couple travel Southeast Asia" returned 50 channels; the first few
  (e.g. Jaime and Adam, Mai and Chris Travel, Cam & Evie Abroad) look like
  close matches to KarmicSteps.
- Test query: 50 results → 17 pass the pre-filter (1k–100k subscribers, ≥20 videos).



In [38]:
import os
from dotenv import load_dotenv
from requests import get
import pandas as pd

## 1. Load the api_key

In [2]:
load_dotenv()
api_key = os.getenv("YOUTUBE_API_KEY")
print("api_key is None:", api_key is None)
print("api_key length:", len(api_key))


api_key is None: False
api_key length: 39


## 2. Call channels.list

In [3]:
URL = "https://www.googleapis.com/youtube/v3/channels"

params = {
    "part": "snippet,contentDetails,statistics",
    "forHandle": "@KarmicSteps",
    "key": api_key,
}

# send a GET request to URL
response = get(URL, params=params, timeout=10)

print("Response status code:", response.status_code)

Response status code: 200


## 3. Explore the JSON

In [4]:
# convert the response from JSON into a Python dict
print("Response status code:", response.status_code)
data = response.json()
print("Top-level keys of data:", data.keys())
if "items" in data:
    print("Top-level keys of first item:", data["items"][0].keys())

Response status code: 200
Top-level keys of data: dict_keys(['kind', 'etag', 'pageInfo', 'items'])
Top-level keys of first item: dict_keys(['kind', 'etag', 'id', 'snippet', 'contentDetails', 'statistics'])


## 4. Pull out the channel data

In [5]:
# Print channel data
channel = data["items"][0]
print("Channel ID:", channel["id"])
print("Channel title:", channel["snippet"]["title"])
print("Subscriber count:", channel["statistics"]["subscriberCount"])
print("Video count:", channel["statistics"]["videoCount"])
print("Uploads playlist ID:", channel["contentDetails"]["relatedPlaylists"]["uploads"])

Channel ID: UCqxY_VZtLNg036hIA-wRahQ
Channel title: Karmic Steps
Subscriber count: 343
Video count: 100
Uploads playlist ID: UUqxY_VZtLNg036hIA-wRahQ


## 5. Check seed channels 

In [ ]:
# define a function get_channel(handle)
def get_channel(handle):
    params = {
        "part": "snippet,statistics,contentDetails",
        "forHandle": handle,
        "key": api_key,
    }
    response = get(URL, params=params, timeout=10)
    
    print("Response status code:", response.status_code)

    data = response.json()
    if "items" not in data:
        return None
    item = data["items"][0]
    return {
        "handle": handle,
        "channel_id": item["id"],
        "title": item["snippet"]["title"],
        "subscribers": int(item["statistics"].get("subscriberCount", 0)),
        "video_count": int(item["statistics"].get("videoCount", 0)),
        "uploads_playlist_id": item["contentDetails"]["relatedPlaylists"]["uploads"]
    }

seeds = pd.read_csv("data/channels_seed.csv")

rows = []
for handle in seeds['handle']:
    results = get_channel(handle)
    if results is None:
        print(f"Warning: handle {handle} not found")
    else:
        rows.append(results)

df = pd.DataFrame(rows)
df = df.sort_values(by='subscribers', ascending=False)
df 

In [20]:
# Filter the seed table
df[(df["subscribers"] >= 1000) & (df["subscribers"] <= 100000)]

,handle,channel_id,title,subscribers,video_count,uploads_playlist_id
3,@AcrossEveryBorder,UCYAIbGWcLdohnpRh-YLXzRw,Sara & Ryan,16900,137,UUYAIbGWcLdohnpRh-YLXzRw
1,@arjayneyra,UCGloEV_XBzi-3Fp2hSnKVWg,Arjay Neyra,4840,466,UUGloEV_XBzi-3Fp2hSnKVWg
8,@TannerandSierra,UCf9kneMiWCVgc47Wzi3ew6A,Tanner and Sierra,4790,600,UUf9kneMiWCVgc47Wzi3ew6A


## 6. Test search.list with one querry

Using the API method `search.list` to search for more channels within the niche. However search list does not return subscriber count. So a second API method `channels.list` and channel ID will be used to retrieve the subscribers, video count and upload playlist.

Search phrases: 
- couple travel vlog
- budget travel tips
- backpacking through South East Asia
- couple travel Southeast Asia
- Van life in Southern Africa
- Southeast Asia backpacking couple (Replacements)
- Korea travel vlog (Replacements)

- ~~Living in Thailand~~: dropped, returned expat/daily-life channels, not travel vlogs
- ~~Living in Korea~~: dropped, same reason, and mostly non-English


In [21]:
# store the search endpoint URL in a variable called SEARCH_URL
SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"

params = {
    "part": "snippet",
    "q": "couple travel Southeast Asia",
    "type": "channel",
    "maxResults": 50,
    "relevanceLanguage": 'en',
    "key": api_key,
}

response = get(SEARCH_URL, params=params, timeout=10)

print("Response status code:", response.status_code)


data = response.json()


Response status code: 200


In [24]:
print(data.keys())
print(len(data["items"]))
print(data["items"][0].keys())
print(data["items"][0]["id"])

dict_keys(['kind', 'etag', 'nextPageToken', 'regionCode', 'pageInfo', 'items'])
50
dict_keys(['kind', 'etag', 'id', 'snippet'])
{'kind': 'youtube#channel', 'channelId': 'UClhQmYpyIedwL-gAW62nF_Q'}


In [35]:
for item in data["items"][:20]:
    print(item["snippet"]["title"])


Warick and Aireen - Island Life
Jaime and Adam
Mai and Chris Travel
Cam & Evie Abroad
TravelGlitchx30
Rhys & Jade
SS Travels | Southeast Asia Guides
Carla and Poppy Travel
MEonthemap | Travel Couple
Joss and Megan Travel 
Surbhi & Chetan
Avoir Travel
Emma & Tyler
SamAndRachTravel
WanderOH - Travel couple Olaf & Hanna
Age Gap No Map
Billie and Matt Travel
Tom And Lauren Travel
Jordan and Matt
Twogether Nomads


In [32]:
first = data["items"][0]
print(first["id"])

print(first["id"]["channelId"])

{'kind': 'youtube#channel', 'channelId': 'UClhQmYpyIedwL-gAW62nF_Q'}
UClhQmYpyIedwL-gAW62nF_Q


In [39]:
channel_ids = []

for i in data["items"]:
    channel_ids.append(i["id"]["channelId"])

print(len(channel_ids))
print(channel_ids[:3])

50
['UClhQmYpyIedwL-gAW62nF_Q', 'UCLk1JGa2AtMqz1fAVFE20Rw', 'UCD9CjkcRJdP_QrQUb7p1srg']


## 7. Batch-look up the 50 results with channels.list and pre-filter

In [42]:
ids_string = ",".join(channel_ids)
print(ids_string[:60])

UClhQmYpyIedwL-gAW62nF_Q,UCLk1JGa2AtMqz1fAVFE20Rw,UCD9CjkcRJ


In [45]:
params = {
    "part": "snippet,statistics,contentDetails",
    "id": ids_string,
    "key": api_key,
}

response = get(URL, params=params, timeout=10)
print("Status code:", response.status_code)

found = response.json()
print("Channels returned:", len(found["items"]))

Status code: 200
Channels returned: 50


In [46]:
found_rows = []

for item in found["items"]:
    found_rows.append({
        "channel_id": item["id"],                           
        "title": item["snippet"]["title"],
        "subscribers": int(item["statistics"].get("subscriberCount", 0)),
        "video_count": int(item["statistics"].get("videoCount", 0)),
        "uploads_playlist_id": item["contentDetails"]["relatedPlaylists"]["uploads"],
    })

found_df = pd.DataFrame(found_rows)
found_df = found_df.sort_values(by="subscribers", ascending=False)
found_df


,channel_id,title,subscribers,video_count,uploads_playlist_id
44,UC4ijq8Cg-8zQKx8OH12dUSw,Kara and Nate,4550000,1061,UU4ijq8Cg-8zQKx8OH12dUSw
37,UCG_bzbQMfRabBhy5cx2sQQg,Retirement Travelers,209000,314,UUG_bzbQMfRabBhy5cx2sQQg
11,UC6MppNglJgO07JrASkSjGKA,BLAG TRAVEL VLOG,40100,674,UU6MppNglJgO07JrASkSjGKA
31,UCpnWPofmXRLBFuteLDXta6w,Ty and Soph,19000,147,UUpnWPofmXRLBFuteLDXta6w
12,UCG5mGwQOmIDAoOW-pLNzwTA,SS Travels | Southeast Asia Guides,9400,10,UUG5mGwQOmIDAoOW-pLNzwTA
14,UC5x9m1sBWVoINBI2qvvu8eQ,One For The Road Travel Vlog,5080,269,UU5x9m1sBWVoINBI2qvvu8eQ
15,UCKG03vEIUFMK21WZ5tmOwDg,TravelingTice,4980,367,UUKG03vEIUFMK21WZ5tmOwDg
13,UCU9TUVtzkibUijF8XB27nbg,Jordan and Matt,3440,32,UUU9TUVtzkibUijF8XB27nbg
7,UCaMtMUIcDIn72QnJBSvGISw,Billie and Matt Travel,3090,0,UUaMtMUIcDIn72QnJBSvGISw
43,UCpLd0xjfUAfhx9QqOpsGnlQ,TwoBritsTravel,3090,339,UUpLd0xjfUAfhx9QqOpsGnlQ


In [47]:
candidates = found_df[
    (found_df["subscribers"] >= 1000) &
    (found_df["subscribers"] <= 100000) &
    (found_df["video_count"] >= 20)
]
print(len(candidates))
candidates


17


,channel_id,title,subscribers,video_count,uploads_playlist_id
11,UC6MppNglJgO07JrASkSjGKA,BLAG TRAVEL VLOG,40100,674,UU6MppNglJgO07JrASkSjGKA
31,UCpnWPofmXRLBFuteLDXta6w,Ty and Soph,19000,147,UUpnWPofmXRLBFuteLDXta6w
14,UC5x9m1sBWVoINBI2qvvu8eQ,One For The Road Travel Vlog,5080,269,UU5x9m1sBWVoINBI2qvvu8eQ
15,UCKG03vEIUFMK21WZ5tmOwDg,TravelingTice,4980,367,UUKG03vEIUFMK21WZ5tmOwDg
13,UCU9TUVtzkibUijF8XB27nbg,Jordan and Matt,3440,32,UUU9TUVtzkibUijF8XB27nbg
43,UCpLd0xjfUAfhx9QqOpsGnlQ,TwoBritsTravel,3090,339,UUpLd0xjfUAfhx9QqOpsGnlQ
32,UCwbDx2UNDbmddeV4rrTOTlA,Rexie & Jas Travels,2980,176,UUwbDx2UNDbmddeV4rrTOTlA
28,UCp_VmQt6wbxECfx0eFivjsQ,Tikkie & Jerry | Beyond Bangkok,2970,381,UUp_VmQt6wbxECfx0eFivjsQ
4,UCmmasDjd2b6Zqv7JcrCEzCg,Bartek and Kesja,2650,130,UUmmasDjd2b6Zqv7JcrCEzCg
29,UCz9GH7exXcW04cz8lyr-B8Q,Claire and Craig Travel,1890,53,UUz9GH7exXcW04cz8lyr-B8Q


## 8. Run all queries and combine the results


In [48]:
queries = [
    "couple travel vlog",
    "budget travel tips",
    "backpacking through South East Asia",
    "Living in Thailand",
    "Living in Korea",
    "couple travel Southeast Asia",
    "Van life in Southern Africa",
]

for query in queries:
    print("Would search:", query)

Would search: couple travel vlog
Would search: budget travel tips
Would search: backpacking through South East Asia
Would search: Living in Thailand
Would search: Living in Korea
Would search: couple travel Southeast Asia
Would search: Van life in Southern Africa
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [49]:
search_rows = []

for query in queries:
    params = {
        "part": "snippet",
        "q": query,
        "type": "channel",
        "maxResults": 50,
        "relevanceLanguage": 'en',
        "key": api_key,
    }
    
    response = get(SEARCH_URL, params=params, timeout=10)    
    print(query, "->", response.status_code)
    results = response.json()

    for item in results["items"]:                      
        search_rows.append({
            "channel_id": item["id"]["channelId"],             
            "found_by": query,                             
        })

print("Total rows:", len(search_rows))

couple travel vlog → 200
budget travel tips → 200
backpacking through South East Asia → 200
Living in Thailand → 200
Living in Korea → 200
couple travel Southeast Asia → 200
Van life in Southern Africa → 200
Total rows: 350


In [50]:
search_df = pd.DataFrame(search_rows)
print("Total rows:", len(search_df))

print("Unique channels:", search_df["channel_id"].nunique())

unique_df = search_df.drop_duplicates(subset= "channel_id")
print("After dropping duplicates:", len(unique_df))

Total rows: 350
Unique channels: 345
After dropping duplicates: 345


In [52]:
unique_df.to_csv("data/search_results.csv", index=False)

In [53]:
check = pd.read_csv("data/search_results.csv")
print(check.shape)
check.head()

(345, 2)


,channel_id,found_by
0,UCiJfplrc7idtWYpDsI325yQ,couple travel vlog
1,UCjzh3Pr_1agR2hP6hKj8dIw,couple travel vlog
2,UCEm59e_lj_y6oP5Bm7l7r3w,couple travel vlog
3,UCa58mLerer3ZIjlDWIGUrsQ,couple travel vlog
4,UCnD3YprKV3KS7zAUkWeK97g,couple travel vlog


## 9. Batch-look up all unique channels

In [51]:
ids = unique_df["channel_id"].tolist() 
print(len(ids))

for start in range(0, len(ids), 50):
    chunk = ids[start:start + 50]
    print(start, len(chunk))

345
0 50
50 50
100 50
150 50
200 50
250 50
300 45


In [54]:
all_channel_rows = []

for start in range(0, len(ids), 50):
    chunk = ids[start:start + 50]
    ids_string = ",".join(chunk)

    params = {
        "part": "snippet,statistics,contentDetails",
        "id": ids_string,
        "key": api_key,
    }
    response = get(URL, params=params, timeout=10)     
    print(start, "->", response.status_code)
    found = response.json()

    for item in found["items"]:
        all_channel_rows.append({
            "channel_id": item["id"],                         
        "title": item["snippet"]["title"],
        "subscribers": int(item["statistics"].get("subscriberCount", 0)),
        "video_count": int(item["statistics"].get("videoCount", 0)),
        "uploads_playlist_id": item["contentDetails"]["relatedPlaylists"]["uploads"],
         
        })

print("Channels looked up:", len(all_channel_rows))

0 -> 200
50 -> 200
100 -> 200
150 -> 200
200 -> 200
250 -> 200
300 -> 200
Channels looked up: 345


In [56]:
channel_df = pd.DataFrame(all_channel_rows)
print(channel_df.shape)
print(channel_df["channel_id"].unique())

(345, 5)
<StringArray>
['UCXSg6Epn-Zighw5WLoebM2A', 'UCwCHVYVkEaGIXno9ArRdtKg',
 'UCnAtqtsHYQK_u7zSJE9h6xw', 'UC_AtDrjxe56R3K4trtUNdQA',
 'UCiJfplrc7idtWYpDsI325yQ', 'UCbn5xzRrAOY9DWJlbyPBsvg',
 'UC8uFoBeb5leM6DKZg6NJopQ', 'UCI86O2Dr9tU60b6CMhOZYNg',
 'UCv6lLCmTzOBNrwchGZxygKw', 'UC7XUu6DQPUnVWujU-SeW9lA',
 ...
 'UCIw7Fsv6FLIUAkpqEQUjQ6A', 'UC1YaffXkummWhrcn6vOttTw',
 'UCm7xoyaTOVmi9V88gSEgMgg', 'UCS5RwPypuxYVNg0n-eEYdmw',
 'UCP-Om90G2k5aoOMjYYyErMQ', 'UC29vLPlafHcsqZu3L-Rk_pA',
 'UClInot6_TbDtMThtBeMtCXA', 'UCkxfaOYOC8W2W_EkCz2e6Kg',
 'UCcURrD9YFNWIRgK7NeibXOQ', 'UCxOutpnz91ehSbIa0ERl27w']
Length: 345, dtype: str


In [57]:
channels_in_range = channels_df[
    (channels_df["subscribers"] >= 1000) &
    (channels_df["subscribers"] <= 100000) &
    (channels_df["video_count"] >= 20)
]
print(len(channels_in_range))
channels_in_range

77


,channel_id,title,subscribers,video_count,uploads_playlist_id
10,UCxnw8U2Kpi9-2POvuAOakYA,Tamil Travel Couple | Indhu Dinesh,4750,179,UUxnw8U2Kpi9-2POvuAOakYA
25,UC2Nik-wcEKzJS4t0JOK5QTQ,Travel with Nomad Couple,2690,622,UU2Nik-wcEKzJS4t0JOK5QTQ
30,UCjzh3Pr_1agR2hP6hKj8dIw,Trip Couple,88000,227,UUjzh3Pr_1agR2hP6hKj8dIw
39,UCEm59e_lj_y6oP5Bm7l7r3w,Travel Couple - SL,25500,303,UUEm59e_lj_y6oP5Bm7l7r3w
41,UC9YUlvBoLFQgfdXBKXkI9sA,Couple Travel Diary,1990,776,UU9YUlvBoLFQgfdXBKXkI9sA
...,...,...,...,...,...
295,UCHeR7lrDEHZuJcjNN9WZKwQ,Larry & Dee,1330,41,UUHeR7lrDEHZuJcjNN9WZKwQ
299,UC5x9m1sBWVoINBI2qvvu8eQ,One For The Road Travel Vlog,5080,269,UU5x9m1sBWVoINBI2qvvu8eQ
316,UCAYItAX-5Ibb_YPSoQ_KORw,Marcius Van,16700,1593,UUAYItAX-5Ibb_YPSoQ_KORw
324,UCiSFjHRNRGNZR0w4VBYARlw,Fin van Wyk,5980,24,UUiSFjHRNRGNZR0w4VBYARlw


In [58]:
channels_in_range = channels_in_range.merge(unique_df, on="channel_id", how="left")
channels_in_range.head()

,channel_id,title,subscribers,video_count,uploads_playlist_id,found_by
0,UCxnw8U2Kpi9-2POvuAOakYA,Tamil Travel Couple | Indhu Dinesh,4750,179,UUxnw8U2Kpi9-2POvuAOakYA,couple travel vlog
1,UC2Nik-wcEKzJS4t0JOK5QTQ,Travel with Nomad Couple,2690,622,UU2Nik-wcEKzJS4t0JOK5QTQ,couple travel vlog
2,UCjzh3Pr_1agR2hP6hKj8dIw,Trip Couple,88000,227,UUjzh3Pr_1agR2hP6hKj8dIw,couple travel vlog
3,UCEm59e_lj_y6oP5Bm7l7r3w,Travel Couple - SL,25500,303,UUEm59e_lj_y6oP5Bm7l7r3w,couple travel vlog
4,UC9YUlvBoLFQgfdXBKXkI9sA,Couple Travel Diary,1990,776,UU9YUlvBoLFQgfdXBKXkI9sA,couple travel vlog


In [59]:
channels_in_range.head()
channels_in_range["found_by"].value_counts()


found_by
Living in Thailand                     29
couple travel Southeast Asia           15
Living in Korea                        10
couple travel vlog                      8
budget travel tips                      6
backpacking through South East Asia     6
Van life in Southern Africa             3
Name: count, dtype: int64

In [60]:
channels_in_range[["title", "found_by"]

,title,found_by
0,Tamil Travel Couple | Indhu Dinesh,couple travel vlog
1,Travel with Nomad Couple,couple travel vlog
2,Trip Couple,couple travel vlog
3,Travel Couple - SL,couple travel vlog
4,Couple Travel Diary,couple travel vlog
...,...,...
72,Larry & Dee,couple travel Southeast Asia
73,One For The Road Travel Vlog,couple travel Southeast Asia
74,Marcius Van,Van life in Southern Africa
75,Fin van Wyk,Van life in Southern Africa


In [66]:
thailand = channels_in_range[channels_in_range["found_by"] != "Living in Thailand"]

In [68]:
channels_in_range = channels_in_range[channels_in_range["found_by"] != "Living in Thailand"]
print(len(channels_in_range))


48


In [65]:
pd.set_option("display.max_rows", 100)

In [69]:
korea = channels_in_range[channels_in_range["found_by"] == "Living in Korea"]
korea[["title", "subscribers"]]


,title,subscribers
49,muril in korea,2200
50,K K Korea Ki ਕਿਰਨ,9410
51,Sush in Korea,90100
52,Our Life In Korea,75400
53,Bibi & Leslie in Korea,3140
54,Amna In Korea,85800
55,My Real Life in Korea,1910
56,iqra Korea,34100
57,Evaa - daily life in Korea,2550
58,Louisa,73100


In [71]:
channels_in_range = channels_in_range[channels_in_range["found_by"] != "Living in Korea"]
print(len(channels_in_range))


38


## 10. Replace the "Living in…" queries

Search phrases: 
- couple travel vlog
- budget travel tips
- backpacking through South East Asia
- couple travel Southeast Asia
- Van life in Southern Africa
- Southeast Asia backpacking couple (Replacements)
- Korea travel vlog (Replacements)
- Thailand travel vlog (new)

- ~~Living in Thailand~~: dropped, returned expat/daily-life channels, not travel vlogs
- ~~Living in Korea~~: dropped, same reason, and mostly non-English


In [72]:
new_queries = [
    "Southeast Asia backpacking couple",
    "Korea travel vlog",
    "Thailand travel vlog",
]

new_rows = []

for query in new_queries:
    params = {
        "part": "snippet",
        "q": query,
        "type": "channel",
        "maxResults": 50,
        "relevanceLanguage": 'en',
        "key": api_key,
    }
    
    response = get(SEARCH_URL, params=params, timeout=10)    
    print(query, "->", response.status_code)
    results = response.json()

    for item in results["items"]:
        new_rows.append({
            "channel_id": item["id"]["channelId"],             
            "found_by": query,                             
        })

print("Total rows:", len(new_rows))

Southeast Asia backpacking couple -> 200
Korea travel vlog -> 200
Thailand travel vlog -> 200
Total rows: 150


In [73]:
# 1. DataFrame + drop duplicates within the new results (same as section 8)
new_df = pd.DataFrame(new_rows)
new_df = new_df.drop_duplicates(subset="channel_id")
print("Unique new results:", len(new_df))

# 2. Save right away, to protect the 300 units
new_df.to_csv("data/search_results_replacements.csv", index=False)

# 3. Drop channels you already looked up in sections 8–9
already_seen = new_df["channel_id"].isin(unique_df["channel_id"])
print("Already seen:", already_seen.sum())

new_df = new_df[~already_seen]
print("Brand-new channels:", len(new_df))

Unique new results: 150
Already seen: 23
Brand-new channels: 127


In [74]:
new_ids = new_df["channel_id"].tolist()
print(len(new_ids))

new_channel_rows = []

for start in range(0, len(new_ids), 50):
    chunk = new_ids[start:start + 50]
    ids_string = ",".join(chunk)

    params = {
        "part": "snippet,statistics,contentDetails",
        "id": ids_string,
        "key": api_key,
    }
    response = get(URL, params=params, timeout=10)     
    print(start, "->", response.status_code)
    found = response.json()

    for item in found["items"]:
        new_channel_rows.append({
            "channel_id": item["id"],                         
        "title": item["snippet"]["title"],
        "subscribers": int(item["statistics"].get("subscriberCount", 0)),
        "video_count": int(item["statistics"].get("videoCount", 0)),
        "uploads_playlist_id": item["contentDetails"]["relatedPlaylists"]["uploads"],
         
        })

print("New channels looked up:", len(new_channel_rows))

127
0 -> 200
50 -> 200
100 -> 200
New channels looked up: 127


In [75]:
# 1. DataFrame (same as section 9)
new_channels_df = pd.DataFrame(new_channel_rows)

# 2. Pre-filter (same three conditions as always)
new_in_range = new_channels_df[
    (new_channels_df["subscribers"] >= 1_000) &
    (new_channels_df["subscribers"] <= 100_000) &
    (new_channels_df["video_count"] >= 20)
]
print("New channels in range:", len(new_in_range))

# 3. Bring back found_by (same merge as before, but from new_df)
new_in_range = new_in_range.merge(new_df, on="channel_id", how="left")

# 4. Stack the new channels under your existing 38
all_candidates = pd.concat([channels_in_range, new_in_range], ignore_index=True)
print("Total candidates:", len(all_candidates))
print("Unique IDs:", all_candidates["channel_id"].nunique())
all_candidates["found_by"].value_counts()


New channels in range: 30
Total candidates: 68
Unique IDs: 68


found_by
Thailand travel vlog                   17
couple travel Southeast Asia           15
couple travel vlog                      8
Korea travel vlog                       8
budget travel tips                      6
backpacking through South East Asia     6
Southeast Asia backpacking couple       5
Van life in Southern Africa             3
Name: count, dtype: int64

## 11. Add seeds and save the final channel list


In [76]:
seeds_in_range = df[
    (df["subscribers"] >= 1_000) &
    (df["subscribers"] <= 100_000) &
    (df["video_count"] >= 20)
]
seeds_in_range = seeds_in_range.drop(columns=["handle"])    
seeds_in_range["found_by"] = "seed"          

channels_final = pd.concat([seeds_in_range, all_candidates], ignore_index=True)
channels_final = channels_final.drop_duplicates(subset="channel_id")

print("Final channels:", len(channels_final))
channels_final["found_by"].value_counts()


Final channels: 71


found_by
Thailand travel vlog                   17
couple travel Southeast Asia           15
couple travel vlog                      8
Korea travel vlog                       8
budget travel tips                      6
backpacking through South East Asia     6
Southeast Asia backpacking couple       5
seed                                    3
Van life in Southern Africa             3
Name: count, dtype: int64

In [77]:
channels_final.to_csv("data/channels.csv", index=False)

check = pd.read_csv("data/channels.csv")
print(check.shape)
check.head()


(71, 6)


,channel_id,title,subscribers,video_count,uploads_playlist_id,found_by
0,UCYAIbGWcLdohnpRh-YLXzRw,Sara & Ryan,16900,137,UUYAIbGWcLdohnpRh-YLXzRw,seed
1,UCGloEV_XBzi-3Fp2hSnKVWg,Arjay Neyra,4840,466,UUGloEV_XBzi-3Fp2hSnKVWg,seed
2,UCf9kneMiWCVgc47Wzi3ew6A,Tanner and Sierra,4790,600,UUf9kneMiWCVgc47Wzi3ew6A,seed
3,UCxnw8U2Kpi9-2POvuAOakYA,Tamil Travel Couple | Indhu Dinesh,4750,179,UUxnw8U2Kpi9-2POvuAOakYA,couple travel vlog
4,UC2Nik-wcEKzJS4t0JOK5QTQ,Travel with Nomad Couple,2690,622,UU2Nik-wcEKzJS4t0JOK5QTQ,couple travel vlog
